##### Copyright 2019 The IREE Authors

In [1]:
#@title Licensed under the Apache License v2.0 with LLVM Exceptions.
# See https://llvm.org/LICENSE.txt for license information.
# SPDX-License-Identifier: Apache-2.0 WITH LLVM-exception

# Low Level Invoke Function

This notebook shows off some concepts of the low level IREE python bindings.

In [2]:
!python -m pip install --pre iree-base-compiler iree-base-runtime -f https://iree.dev/pip-release-links.html

Looking in links: https://iree.dev/pip-release-links.html


In [3]:
import numpy as np

from iree import runtime as ireert
from iree.compiler import compile_str

In [4]:
# Compile a module.
SIMPLE_MUL_ASM = """
  module @arithmetic {
    func.func @simple_mul(%arg0: tensor<4xf32>, %arg1: tensor<4xf32>) -> tensor<4xf32> {
      %0 = arith.mulf %arg0, %arg1 : tensor<4xf32>
      return %0 : tensor<4xf32>
    }
  }
"""

# Compile using the vmvx (reference) target:
compiled_flatbuffer = compile_str(SIMPLE_MUL_ASM, target_backends=["vmvx"])

In [5]:
# Register the module with a runtime context.
# Use the "local-task" CPU driver, which can load the vmvx executable:
config = ireert.Config("local-task")
ctx = ireert.SystemContext(config=config)
vm_module = ireert.VmModule.from_flatbuffer(ctx.instance, compiled_flatbuffer)
ctx.add_vm_module(vm_module)

# Invoke the function and print the result.
print("INVOKE simple_mul")
arg0 = np.array([1., 2., 3., 4.], dtype=np.float32)
arg1 = np.array([4., 5., 6., 7.], dtype=np.float32)
f = ctx.modules.arithmetic["simple_mul"]
results = f(arg0, arg1).to_host()
print("Results:", results)

INVOKE simple_mul
Results: [ 4. 10. 18. 28.]


/tmp/ipykernel_2945214/2849083606.py:5: UserWarning: Making copy of unaligned VmModule buffer. It is recommended to make this deterministic by calling `copy_buffer` to always make a copy or `mmap` to efficiently load from a file. This warning can be silenced by adding `warn_if_copy=False` to `from_buffer`
  vm_module = ireert.VmModule.from_flatbuffer(ctx.instance, compiled_flatbuffer)
